---
authors:
  - edesz
date: 2025-10-04
---

# Estimate Cohort Size Using Predicted ROI

## About

In this step, we will use the best ML model's prediction probabilities and an estimate of the Return on Investment (ROI) from targeting at-risk customers to identify this cohort for two assumed budget scenarios.

### Use-Case Context

As mentioned in the [project scope](../references/02_scope.md), the dataset provided to us by the client represents historical customer behavior where the churn outcome is already known. So, the purpose of the trained ML model is not to act on these historical customers, but to simulate how it would perform if deployed on similar customers in practice. By applying the model to all customers and ranking them by predicted risk of churn (`y_pred_proba`), we can estimate the financial impact if the client had proactively targeted the top-N at-risk customers at that time. The resulting analysis is therefore retrospective since we are performing it on historical customers but forward-looking in its intent, as it estimates the expected business impact of deploying the model on future customers with similar characteristics (features).

This allows the client (credit card division manager) to determine an optimal targeting strategy under realistic assumptions about targeting (intervention) cost and effectiveness. So, the analysis in this notebook will identify this cohort by selecting the customers who most efficiently maximize the net savings out of all ~10,000 customers.

Finally, as [discussed regarding reporting metrics](../references/scope/07_reporting_metrics.md#reporting-budget-scenarios), we assume two scenarios about the client's available budget and make recommendations based for each scenario.

### Relevant Assumptions

We will use the same assumptions made during the project scoping

1. intervention (targeting) success rate of 40%
2. intervention (targeting) cost of $50 per churned customer
3. three sources of fee revenue earned from each customer based on
   - number of credit card transacitons (assumed to be 2%)
   - credit card balance (assumed to be 18%, compared to 15-20% in Canada and 20-30% in the U.S.)
   - credit card exposure (fees determined based on category of card; see the `Card_category` column of the data)
4. other terms
   - loyalty discount factor of 0.9 (or 10% per year)
   - expected remaining tenure of 3 years
5. future customers behave like those characterized by this dataset of ~10,100 customers data provided to us by the client for use in this project

Please see the project scope for more details about how these assumptions are used to calcuate Customer Lifetime Value (CLV), net savings and ROI. Here, a custom Python function `get_costs()` is defined in `src/cc_churn/costs.py` to estimate ROI based on customer attributes from the data and using the above assumptions.

:::{note} Outputs
Charts will be saved as `.html` files in `reports/figures`.

Based on the deliverables [in this project's scoping document](https://github.com/edesz/credit-card-churn/blob/main/references/08_deliverables.md), this notebook exports a single file to the R2 bucket with the at-risk customers (i.e. the cohort), their features, their predicted probaility (to churn) and their business metrics for each of the two assumed scenarions described above.

The filenames will contain the word `roi`.
:::

## Python Imports

The required Python modules are imported below

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown

Define the path to the project root directory

In [ ]:
PROJ_ROOT = Path.cwd().parent

Load environment variables with secrets for use in `boto3`

In [ ]:
assert load_dotenv(dotenv_path=PROJ_ROOT.parent / ".env")

Import the required custom modules

In [ ]:
import cc_churn.costs as costs
import cc_churn.visualization as vzu
import r2.io_utils as r2io
from utils.df_utils import show_df

## User Inputs

Below are the variables that will be used later in this step

In [ ]:
# columns to load
columns = [
    "clientnum",
    "card_category",
    "total_revolv_bal",
    "total_trans_amt",
    "model_name",
    "y_pred_proba",
    "y_pred",
    "best_decision_threshold",
    "is_churned",
]

# costs
# # revenue from transactions (bank earns #% of transaction volume)
interchange_rate = 0.02
# # revenue from revolving balance (~20% interest)
apr = 0.18
# # fee revenue from credit card exposure (modeled from card type)
card_fees = {"Blue": 0, "Silver": 50, "Gold": 100, "Platinum": 200}
tenure_years = 3
discount = 0.9
# # percentage of churners who can be convinced to stay (i.e. success rate
# # of saving a churning customer)
success_rate = 0.40
# # cost of intervention to get a single customer to not churn (discounts,
# # call center time, retention offers, etc.)
intervention_cost = 50
# # maximum number of customers that can be targeted based on client's budget
num_customers_max = 100

# predictions prefix
# # folder containing predictions
prefix = "cloud-run"
# # prefix of filename with predictions
r2_key_pred = "all_predictions__"

Use environment variables to define an authenticated `boto3` R2 client and define the CLV multiplier [as per the project scope](../references/scope/02_costs.md#clv-multiplier)

In [ ]:
reports_dir = PROJ_ROOT / "reports"
figures_dir = reports_dir / "figures"

account_id = os.getenv("ACCOUNT_ID")
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
bucket_name = os.getenv("BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    region_name="auto",
)

# costs
multiplier = (1 - discount**tenure_years) / (1 - discount)

## Load Data with Predictions

Load predictions for all available customers

In [ ]:
df_all_pred = r2io.pandas_read_latest_parquet_r2(
    s3_client,
    bucket_name,
    f"{prefix}/",
    r2_key_pred,
    ".parquet.gzip",
    columns,
).astype({"card_category": "category", "model_name": "category"})

Use model predictions of all data to extract best decision threshold

In [ ]:
best_decision_threshold = (
    df_all_pred["best_decision_threshold"].head(1).squeeze()
)

Extract name of best ML model from model predictions

In [ ]:
best_model_name = df_all_pred["model_name"].head(1).squeeze()

## Estimate Business Metrics

Calculate business metrics from targeting all customers predicted to churn (i.e. `y_pred == 1`)

In [ ]:
%%time
df_business_metrics, _, _ = costs.get_cost(
    df_all_pred,
    best_decision_threshold,
    interchange_rate,
    apr,
    card_fees,
    multiplier,
    success_rate,
    intervention_cost,
)
_ = show_df(df_business_metrics)
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(df_business_metrics.head(1))

:::{hint}
In the above, `get_cost()` calls two functions to estimate savings: `calc_predicted_savings()` and `calc_true_savings()` for estimating the predicted and true savings in `expected_savings` and `true_savings` respectively. The `clv` (predicted CLV) is also returned.

Next, the customers are sorted in descending order of `y_pred_proba` and the cumulative predicted and true savings are calculated as the number of customers (*N*) is increased. Finally, the total intervention cost (assumed to be 50 dollars per customer) is calculated. Using these inputs, we get the true and predicted ROI for the top *N* customers is estimated as the ratio of the predicted or true cumulative savings to the total intervention cost.
:::

**Notes**

1. The following metrics are estimated per customer in the data, and are discussed in the [project scoping document](../references/scope/02_costs.md)
   - `interchange_rev`
     - revenue earned from number of credit card transactions performed, earned from *n*th customer
   - `interest_rev`
     - revenue from credit card interest due to a credit card balance, earned from *n*th customer
   - `fee_rev`
     - revenue from credit card exposure, earned from *n*th customer
   - `annual_rev`
     - annual revenue earned from *n*th customer
   - `clv`
     - CLV of *n*th customer
   - `expected_savings`
     - predicted savings from targeting the *n*th customer
   - `true_savings`
     - true savings from targeting the *n*th customer
2. The following columns show cumulative inputs needed to calculate the return on investment after targeting the top customers
   - `n`
     - number of at-risk customers to target
     - customers are sorted in reverse chronological order of prediction probability and then ranked, starting at 1
       - `n` is the rank
   - `total_intervention_cost`
     - total cost to client to intervene to try to get single customer to revert their decision to churn
   - `cum_pred_savings`
     - cumulative prediced savings after targeting `n` at-risk customers
   - `cum_true_savings`
     - cumulative true savings after targeting `n` at-risk customers
   - `ROI`
     - true return on investment from targeting at-risk customers (`n`)
   - `ROI_pred`
     - predicted return on investment from targeting at-risk customers
   - `ROI_error`
     - error in predicted return on investment from targeting at-risk customers
   - `ROI_percent`
     - true ROI expressed as a percentage
     - better than `ROI` for reporting
   - `ROI_percent_pred`
     - predicted ROI expressed as a percentage
     - better than `ROI_pred` for reporting
2. The business metrics are only calculated for customers that are predicted to churn since these are the customers that are candidates for targeting by the client's team. For this reason, there are fewer rows in the business metrics `DataFrame` (`df_business_metrics`) than in the `DataFrame` with the ML model predictions for all available data (`df_all_pred`).

## Estimate Cohort Size - Optimal Number of Customers to Target (`N`)

### Plot Cumulative ROI versus Number of Predicted Churners

Plot true and predicted expected savings and ROI curves to visualize the following

1. true ROI
2. predicted ROI

using all available data

In [ ]:
%%time
vzu.plot_roi_curves(
    df_business_metrics['n'],
    df_business_metrics['ROI_percent'],
    df_business_metrics['ROI_percent_pred'],
    ptitle=(
        'Selecting Top ~400 At-Risk Customers Gives Fastest ROI Growth, '
        'Reaching ~150% ROI'
    ),
    legend_loc='lower right',
    xlabel=f"Number of Predicted Churners to Contact (Sorted)",
    ylabel="ROI(%)",
    grid_line_opacity=0.25,
    ticklabel_font_size=14,
    axis_label_font_size=14,
    title_font_size=14,
    axis_font_color='#454545',
    line_colors={'Predicted': 'darkgreen', 'True': '#cccccc'},
    fig_size=(12, 6),
)

Generally, as more top at-risk customers are targeted and successfully reverted the ROI also increases. This makes sense since we are recovering lost credit card revenue from these customers if they decide not to cancel their services at the bank.

We can see there are three regions in this chart.

The first region (called the *top 400* region) occurs between 0 and approximately 400. The top 400 at-risk customers are found here. Excluding an initial noisy period at the beginning, corresponding to the top 50 customers, the ROI increases fastest of the three regions from 0 to ~150%.

The second region (*middle dip*) shows a dip in the ROI. This occurs when the top 400 to the top 560 customers are selected. As more at-risk customers are selected in this region, the ROI decreases instead of increasing, as it did in the *top 400* region. This is not expected.

Finally, in the third region (*tail*), the ROI increases again as more customers are selected, but at a slower rate than in the first region.

Below, we add columns to the business metrics data that contains the

1. name of the region for each predicted churner among all existing customers
2. check if row is a false positive

In [ ]:
df_business_metrics = df_business_metrics.assign(
    region=lambda df: (
        pd.cut(
            df["n"],
            bins=[1, 400, 560, len(df)],
            labels=["top_400", "middle_dip_region", "tail_region"],
            include_lowest=True,
            right=True,
        )
    ),
    is_false_positive=lambda df: (df["y_pred"] == 1) & (df["is_churned"] == 0),
)

### Characterize Regions of the Chart

Extract the following business metrics and metadata per region

1. start of each region (`n_start`)
2. end of each region (`n_end`)
3. number of customers in each region (`n_customers`)
4. average predicted churn rate (`churn_rate`)
5. average prediction probability (`avg_proba`)
6. minimum prediction probability (`min_proba`)
7. average false positive rate (`avg_fpr`)
7. average predicted CLV (`avg_clv`)
8. average predicted savings (`avg_expected_savings_per_customer`)

In [ ]:
df_regions_business_ml_metrics = df_business_metrics.groupby("region").agg(
    n_start=("n", "min"),
    n_end=("n", "max"),
    n_customers=("n", "count"),
    churn_rate=("is_churned", "mean"),
    avg_proba=("y_pred_proba", "mean"),
    min_proba=("y_pred_proba", "min"),
    avg_fpr=("is_false_positive", "mean"),
    avg_clv=("clv", "mean"),
    avg_expected_savings_per_customer=("expected_savings", "mean"),
)

Extract the correlation between predicted probability and predicted savings (`proba_vs_expected_savings_corr`) per region

In [ ]:
df_proba_savings_corr = (
    df_business_metrics.groupby("region")
    .apply(lambda x: x["y_pred_proba"].corr(x["expected_savings"]))
    .rename("proba_vs_expected_savings_corr")
)

Combine all metrics per region

In [ ]:
df_n_regions_summary = pd.concat(
    [df_regions_business_ml_metrics, df_proba_savings_corr],
    axis=1,
).transpose()

df_n_regions_summary.style.apply(
    lambda x: ["background-color: yellow"] * len(x),
    axis=1,
    subset=(
        [
            "churn_rate",
            "avg_proba",
            "min_proba",
            "avg_fpr",
            "avg_clv",
            "avg_expected_savings_per_customer",
            "proba_vs_expected_savings_corr",
        ],
        ["middle_dip_region"],
    ),
)

In the *middle dip* region, we can observe the following

1. The ROI depends on three terms: CLV, success rate and cost. The latter two terms are assumed to be fixed. Average CLV (`avg_clv`) is lower than in the *top 400* region. With a lower CLV, the ROI is lower as is observed here. This suggests lower-value customers are being picked up in this region.
2. Average predicted savings (`avg_expected_savings_per_customer`) is also lower, which suggests these customers, if successfully reverted, make a weaker contribution to ROI.
3. Average predicted churn rate (`churn_rate`) is also slightly lower in the *middle dip* region, which is confirmed by the higher false positive (FP) rate (`avg_fpr`) here. FPR is an order of magnitude higher in the middle dip region than in the top 400 region.
4. Customers are ranked by prediction probability. The higher the probability the higher the rank. The correlation between true savings and predicted probabilities is not high in the *top 400* region since the top ranked customers by predicted churn probability are a mixture of low- and high-value customers in this random sample of data being used in this project. The correlation is high for the high-value customers only. The presence of more high-value customers leads to more true savings.

    Based on the `proba_vs_expected_savings_corr` column, the correlation between predicted probability and true savings is significantly weaker in the *middle dip* region than in the *top 400* region. This suggests more lower-value customers are placed here by their ranking. This is in line with the observation of a lower CLV observed here. However, the average and minimum predicted probabilities (`avg_proba` and `min_proba`) have not changed much between the top two regions. This suggests there might be a problem with the ranking logic in this region.

    This problem would be caused by lower model accuracy in this region than in in the *top 400* region. This is likely explained by the order of magnitude increase in FPR in this region. So, the lower-value customers who were incorrectly placed here by the model should not be in this region and are contributing to a lower ROI here.

If we did target customers in this *middle* region and successfully retain them then the relative benefit compared to the *top 400* region is smaller. As seen earlier, the intervention cost per customer is assumed to be fixed at $50 per customer. As a result, the ROI is also smaller here compared to the initial region.

Overall, the lower CLV, lower average expected savings and lower correlation between probability and savings confirm that the dip is caused by a transition zone in which the model incorrectly ranks customers as high-risk, but their economic value is weaker or noisier, reducing ROI efficiency.

From a marginal ROI perspective, it seems that it would be optimal to target the top ~400 customers. This segment captures the highest concentration of expected value per intervention, where the model's predicted probability, average CLV, and intervention effectiveness (combination of our assumed intervention cost and success rate) combine favourably and maximize the predicted ROI. In this region, each additional customer contributes substantial incremental net savings, causing the predicted ROI to rapidly increase from 0 to ~150%.

We walked through the problems with the second region (*middle dip region*), so customers in this region should be avoided.

In the third region (*tail region*), the model begins to include customers with a lower average churn risk (`min_proba`) and the slope of the ROI curve starts flattening. So, the marginal benefit of each additional customer drops. At the same, costs increase linearly since we assumed a fixed cost per customer of $50. Combined, this results in diminishing returns in this region. So, although total ROI continues to increase up to ~250%, it requires roughly three times more customers (~1,200) to achieve that gain.

With this in mind, ~400 customers (i.e. the *top 400* region) represents the most efficient allocation of resources, maximizing ROI growth per unit cost and delivering ROI the fastest without evidence of diminishing returns.

Note that we're implicitly optimizing *ROI efficiency* by targeting customers in the first (*top 400*) region, and not total profit or ROI. If instead the client cared about total net savings (`cum_pred_savings` in `df_business_metrics`) then the optimal `N` would be closer to ~1200.

### Estimating Cohort Size by Comparing Two Budget Scenarios

Depending on the client's available budget, there are two possible recommendation scenarios

1. budget of at least approximately 20,000 dollars: if the client has a constraint in the budget that requires at most 400 customers or less (at an assumed cost of $50 per customer) then we would recommend to target the top 400 at-risk customers since this cohort *optimizes ROI efficiency* and returns a ROI of ~150%
2. budget of at least approximately 80,000 dollars: if the budget is higher and allows for up to 1,600 customers to be targeted then the client can *optimize ROI* by targeting all customers predicted to be at risk of churning, achieving a ROI of ~250%

We'll now programmatically extract the optimal number of customers to target as well as the relevant business metrics per scenario.

#### Scenario 1 - Constrained by Intervention Budget of At Least $20,000

We'll first extract the cohort size based on scenario one (target at most the top 400 at-risk customers)

In [ ]:
df_business_metrics_optimal_constrained = (
    df_business_metrics.query(
        "(total_intervention_cost > 0) & "
        # capture first region in ROI
        f"(n <= 400)"
    )
    .sort_values(
        by=["ROI", "n"],
        ascending=[False, True],
        ignore_index=True,
    )
    .head(1)
)
optimal_N_roi_constrained = (
    df_business_metrics_optimal_constrained["n"].squeeze()
)

#### Scenario 2 - Higher Budget of At Least $80,000

For the second scenario a higher budget is available that allows for targeting all customers predicted to be at risk of churning, so we recommend the client target all at-risk customers

In [ ]:
df_business_metrics_optimal_no_constrain = (
    df_business_metrics.tail(1).reset_index(drop=True)
)
optimal_N_roi_no_constrain = (
    df_business_metrics_optimal_no_constrain["n"].squeeze()
)

#### Comparison

Below is a summary of the business metrics for the two scenarios

In [ ]:
(
    pd.concat(
        [
            df_business_metrics_optimal_constrained.assign(scenario=1),
            df_business_metrics_optimal_no_constrain.assign(scenario=2),
        ],
        ignore_index=True,
    )[
        [
            "n",
            "total_intervention_cost",
            "cum_true_savings",
            "cum_pred_savings",
            "ROI_error",
            "ROI_percent",
            "ROI_percent_pred",
            "scenario",
        ]
    ].style.set_properties(
        subset=["ROI_error", "ROI_percent_pred"],
        **{"background-color": "yellow", "color": "black"},
    )
)

The error in the predicted ROIs are similar to each other and are negligible as they are both below 2%.

Scenario 2 requires the client to

1. target four times more customers (~1,600 vs ~400)
2. spend four times more on total customer intervention costs (~80,000 vs ~20,000 dollars)

than scenario 1.

Scenario 2 delivers and estimate for

1. savings that are higher by a factor of 6.7 (~200,000 vs ~30,000 dollars)
2. ROI (from the model's predictions) that is higher by a factor of 1.65 (~250% vs ~150%)

than scenario 1.

Scenario 2 optimizes ROI. Scenario 1 optimizes *ROI efficiency*.

### Append Metadata to Business Metrics

Before exporting to disk, we will append two columns of metadata to the business metrics

1. `maximizes_roi` to indicate if targeting customer maximizes ROI for each scenario
2. (for convenience) copy of `y_pred` renamed to `is_at_risk` since it indicates if a customer is at-risk (1) or not (0)

In [ ]:
df_business_metrics = df_business_metrics.assign(
    maximizes_roi_scenario_1=lambda df: df["n"] <= optimal_N_roi_constrained,
    maximizes_roi_scenario_2=lambda df: df["n"] <= optimal_N_roi_no_constrain,
    is_at_risk=lambda df: df["y_pred"],
)

## Export Project Deliverables to Private R2 Bucket

Get the current timestamp in the format `YYmmdd_HHMMSS`

In [ ]:
curr_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

Next, export to a file in the R2 bucket with the following file name format `at_risk_customers_with_business_metrics__<best-model-name>__<current-timestamp-YYmmdd_HHMMSS>.parquet.gzip`

In [ ]:
# %%time
# key_prefix = r2_key_pred.split("/")[0]
# r2io.export_df_to_r2(
#     s3_client=s3_client,
#     df=df_business_metrics,
#     bucket_name=bucket_name,
#     r2_key=(
#         f"{key_prefix}/at_risk_customers_with_business_metrics__"
#         f"{best_model_name.lower()}__"
#         f"{curr_timestamp}.parquet.gzip"
#     ),
#     verbose=False,
# )

## Conclusion

We found three regions in total predicted ROI as more at-risk customers are targeted.

If a budget is available of approximately 80,000 dollars, then all ~1,600 at-risk customers can be targeted. We estimate this will deliver an ROI of ~250% and is within 2% of the true ROI. This cohort just optimizes *ROI*.

The region with the *top 400* at-risk customers represents the best estimate of the cohort that the client should target if the budget is approximately 20,000 dolllars. We estimate the client can realize an ROI of approximately 150%, and is within 2% of the true ROI. This cohort optimizes *ROI efficiency*.

In order for these estimates to be relevant to other customers, not included in the random sample we used in this project, the other sample must have the same characteristics as those seen in the customers whose data was used in the analysis here. In a later step, we develop a data validation model that can be used to validate customer data before being used with the model developed here.